# Systems Design: Voice Interview Platform at Scale

The whiteboard question: voice call → transcript → themes → dashboard, for thousands of parallel interviews.

This notebook is an original tutorial extending the course. It is the
capstone that composes notebooks 22-26 into one system, in the shape of a
systems-design interview answer (2026 state of the art).

## Learning Objectives

- Draw the two-plane architecture (real-time plane vs async analysis plane) and say where every piece of state lives.
- Scale to thousands of parallel interviews and identify the *real* bottleneck (provider concurrency quotas).
- Reason about cost/latency/quality: fast models in the loop, frontier models offline; cost per interview-minute.
- Handle a dropped connection mid-interview end to end.
- Roll out a model swap safely: shadow → canary → ramp → full, with study-level A/B.

## Mental Model

Design it as **two decoupled planes meeting at the transcript store**:

- The **real-time plane** is latency-critical (sub-1s turns), stateful,
  sticky, and expensive to get wrong in the moment.
- The **async plane** is throughput-oriented, trivially retryable, and where
  quality (and the product's value) lives.

And one framing sentence worth saying out loud in the interview: *"an
interview is a 30-minute distributed transaction with a human who won't
retry — never lose data; always be resumable."*

```
[Participant: browser WebRTC / phone via Twilio SIP trunk]
        ▼
[Media layer: LiveKit SFU]  ← agent worker joins room
        ▼
[Agent worker: VAD → STT stream → LLM (fast tier) + guide state machine → TTS stream]
        │  turn-final transcripts → queue + Postgres (append-only), audio → S3 egress
        ▼
[Transcript & session store: Postgres (durable) / Redis (hot) / S3 (audio)]
        │  "interview.completed" event
        ▼
[Async pipeline: re-transcribe (batch, diarized) → per-interview extraction
 → embed → cluster → taxonomy → classify → per-theme synthesis]
        ▼
[Dashboard: counts, confidence tiers, quotes with audio timestamps]
```

## Important Concepts

- **Two planes**: real-time (voice loop) vs async (analysis); they only share the transcript store.
- **Guide as state machine**: sections, objectives, probe budget per question; the LLM decides transitions — not a freeform mega-prompt. Gives determinism, auditability, resumability.
- **State placement**: worker process (sticky session), Redis (per-turn checkpoint), Postgres (append-only turns), S3 (audio via parallel egress).
- **Provider concurrency as the bottleneck**: Deepgram ~50-150 concurrent streams default, ElevenLabs 20-30 concurrent, LLM TPM/RPM tiers.
- **Degradation ladder**: fallback provider → cheaper model → voice-to-text-chat → save partial + resume link.
- **Four-stage rollout**: shadow → canary (new sessions only) → ramp → full; A/B randomized at *study* level.

## Need-To-Know Coverage Checklist

- [x] Full pipeline sketch with queues, workers, and state placement.
- [x] Scaling: sticky sessions, graceful drain, warm pools, pre-scaling for scheduled studies, admission control.
- [x] Provider quota math and mitigations (enterprise contracts, multi-provider sharding, token buckets).
- [x] Cost per interview-minute (~$0.06-0.19 cascaded) and why inference is <5% of all-in cost.
- [x] Model tiers: fast in the loop, frontier offline; prompt caching on the growing-transcript prefix.
- [x] Dropped connection: ICE restart → checkpoint rehydration → persist-then-callback; partial-transcript salvage.
- [x] Model swap: abstraction layer, prompt registry, eval-gated CI with simulated participants, shadow/canary.
- [x] The real companies: Listen Labs, Outset, Strella, Arbor — differentiation is product-layer, not voice plumbing.

## Deep Study Notes

### Scaling to thousands of parallel interviews

- **Unit of scale = agent worker.** Workers are I/O-bound orchestrators
  (~0.1-0.25 vCPU per session); media compute lives on the SFU, inference on
  providers. One worker handles dozens of sessions.
- **Sessions are sticky and long (10-45 min)** → scale-in must drain
  gracefully; aggressive downscaling is wrong. HPA on multi-metric (CPU +
  memory + queued-job depth). Keep a warm pool (2-3 idle workers/region).
- **Interviews are scheduled** — you know load in advance. Pre-scale before a
  2,000-participant study launches; use admission control (waiting room,
  "starts in 30s") rather than failing joins.
- **The real bottleneck is provider quotas**, not your fleet: Deepgram
  streaming defaults ~50-150 concurrent WebSockets per project; ElevenLabs
  20-30 concurrent below Enterprise; thousands of sessions ≈ tens of
  thousands of LLM RPM. Mitigations: enterprise contracts with reserved
  concurrency, multi-provider and multi-account sharding behind a gateway,
  per-study concurrency caps, client-side token buckets that shed to fallback
  providers *before* 429s, queueing interviews when saturated.

### Cost per interview-minute (cascaded, 2026)

| Component | $/min |
|---|---|
| Streaming STT (Deepgram Nova-3) | ~$0.005 |
| LLM (fast tier, per turn) | $0.01-0.04 |
| TTS (Cartesia / ElevenLabs) | $0.03-0.08 |
| Media/infra | $0.005-0.02 |
| **Total** | **~$0.06-0.19/min** |

A 30-min interview ≈ **$2-6 of inference+media** — vs a $75-150 participant
incentive and hundreds of dollars for a human moderator. Inference is <5% of
all-in cost, which is *why the business works* and why you can afford quality
headroom. Prompt caching (system prompt + guide + growing transcript is a
textbook cache-friendly prefix) cuts loop LLM cost 50-90%.

### Model selection

- **In the loop**: fast tier (Haiku/Flash-class, 300-800ms TTFT budget).
  Latency beats IQ here — a brilliant-but-2s model *feels* worse than a
  good-500ms model.
- **Offline**: frontier tier; a 30-min transcript (~7k tokens) with multi-pass
  analysis costs cents-to-$1 per interview even on frontier models.
- **Routing within the loop**: cheap model for acknowledgments and guide
  advancement; run the "should I probe deeper?" planner asynchronously while
  the fast model handles the immediate turn; filler phrases mask tool latency.
- **Cascaded vs S2S**: cascaded remains the production default — transparent,
  debuggable, component-swappable, and you get the transcript for free (the
  transcript *is* the product). S2S has better prosody but weaker
  steerability for long structured interviews and harder evals.

### Dropped connection mid-interview (the set-piece answer)

1. **Blip (<15-30s)**: WebRTC ICE restart; LiveKit auto-reconnects and
   resumes the same session — participant hears a pause.
2. **Hard drop**: state (transcript, guide position, covered topics) was
   checkpointed to Redis/Postgres *after every turn*, so a rejoin — even on a
   different worker — rehydrates and resumes: "Welcome back, we were talking
   about X." Resume window ~15 min; resume if >~20% complete; never make a
   human restart 20 minutes of interview.
3. **PSTN drop**: no resume exists — persist-then-callback: detect the BYE/
   timeout, persist state, call back, open with "looks like we got cut off."
4. **Salvage**: turns are append-only and written in-stream, so a crash loses
   at most the in-flight utterance; re-transcribe the tail from the parallel
   S3 audio recording. Partial interviews still flow to analysis, flagged —
   at n=500, partial data has value.
5. **Agent crash**: supervisor restarts the job against the same room; state
   rehydrates from checkpoint.

### Provider outages: the degradation ladder

Fallback adapters at every stage (primary/secondary STT, TTS with a
similar-sounding fallback voice, LLM across providers), triggered on timeout/
5xx/429/mid-stream disconnect, wrapped in circuit breakers so you fail over
fleet-wide. Ladder: (1) same-quality fallback provider → (2) cheaper model +
canned transitions → (3) degrade modality: voice → text chat in-session →
(4) politely end, save partial, auto-schedule resume.

### Swapping models without breaking the product

1. **Gateway abstraction** — swap is a config change.
2. **Prompt registry with semver** — prompt+model+params = an immutable
   "conversation policy"; every session logs its version.
3. **Eval-gated CI** — golden transcripts plus **simulated participants**
   (an LLM plays the persona; a judge scores probing depth, guide coverage,
   no leading questions).
4. **Shadow traffic** — replay live turn contexts against the candidate;
   judge both. (Caveat to say out loud: shadow can't test interactivity —
   a different follow-up would change the conversation — hence simulation
   and canary.)
5. **Canary** — 1-5% of *new sessions* (never switch mid-interview); auto-
   rollback on completion rate, duration, satisfaction, interruption rate,
   downstream insight-quality deltas.
6. **A/B at study level** — interviews within a study must be comparable; a
   mid-study model change corrupts the research data itself. This
   domain-specific point scores heavily at the whiteboard.

### The market (context for the interview)

Listen Labs (scale-qual flagship, thousands of parallel interviews per
study), Outset (self-serve, probing-depth control), Strella (deterministic
guide scripts, live listen-in, chat→video escalation), Arbor (newer, same
shape). None publish architecture; all fit the reference design above.
Differentiation is **product-layer** — guide determinism, probing control,
synthesis quality, fraud detection — because the voice plumbing is
commoditized by LiveKit/Pipecat.

## Common Failure Modes

- One plane: analysis coupled to the live call → an analysis bug degrades live latency.
- Interview guide as one mega-prompt → non-deterministic coverage, impossible to audit or resume.
- State only in the worker → any crash loses a 25-minute interview.
- Scaling on CPU only → memory pressure from session state kills workers first.
- Ignoring provider quotas until launch day → a 2,000-participant study 429s at interview #151.
- Switching models mid-study → research data no longer comparable; customer trust gone.

## Implementation Notes

- Say the two-plane split and the state-placement table *first* at a whiteboard; details hang off that skeleton.
- Volunteer the quota bottleneck unprompted — it separates candidates who have operated these systems from those who have read about them.
- Keep the cheat-sheet numbers ready: <1s voice-to-voice; $0.06-0.19/min; Deepgram ~50-150 streams; ElevenLabs 20-30 concurrent; shadow → 1-5% canary → ramp → full.

In [ ]:
"""Capacity + cost model for a 2,000-participant study launch.

The napkin math an interviewer wants to see: given provider quotas and a
study size, how many interviews can run concurrently, how long does the
study take, and what does it cost?
"""
import math

STUDY = {"participants": 2000, "interview_minutes": 30}

QUOTAS = {  # effective concurrent-session caps per provider account
    "deepgram_stt_streams": 150,
    "elevenlabs_tts_concurrent": 30,   # <-- the surprise bottleneck
    "llm_effective_sessions": 500,
    "worker_fleet_sessions": 1000,
}

COST_PER_MIN = {"stt": 0.005, "llm": 0.02, "tts": 0.05, "infra": 0.01}


def plan(study, quotas, tts_accounts=1):
    quotas = {**quotas,
              "elevenlabs_tts_concurrent": quotas["elevenlabs_tts_concurrent"] * tts_accounts}
    bottleneck = min(quotas, key=quotas.get)
    concurrency = quotas[bottleneck]
    waves = math.ceil(study["participants"] / concurrency)
    wall_clock_h = waves * study["interview_minutes"] / 60
    total_min = study["participants"] * study["interview_minutes"]
    cost = total_min * sum(COST_PER_MIN.values())
    return {"bottleneck": bottleneck, "concurrency": concurrency,
            "waves": waves, "wall_clock_hours": round(wall_clock_h, 1),
            "inference_cost_usd": round(cost)}


print("single TTS account:", plan(STUDY, QUOTAS))
print("sharded 5 TTS accounts:", plan(STUDY, QUOTAS, tts_accounts=5))
# Note the shape of the answer: sharding TTS moves the bottleneck to STT —
# capacity planning is iterative whack-a-mole across providers.


## Practice

1. Extend `plan()` to model a degradation ladder: if TTS quota saturates,
   overflow sessions fall back to a second provider with different cost and
   quota. How does wall-clock change?
2. Whiteboard (on paper) the full diagram from memory: two planes, state
   placement, the queue between them, and the dashboard.
3. You must swap the conversational LLM two days before a customer's flagship
   study. Walk the four-stage rollout and state what you do about the
   in-flight study specifically.
4. Estimate: at $0.10/min inference and a $100 incentive per participant,
   what fraction of a 30-minute interview's cost is inference? What does that
   imply about where to spend engineering effort?

## Design Checklist

- [ ] Two planes, decoupled at the transcript store.
- [ ] Guide modeled as a state machine; every session logs its policy version.
- [ ] State checkpointed per turn; any worker can resume any session.
- [ ] Audio recorded via parallel egress, never in the hot path.
- [ ] Provider quotas mapped before launch; sharding + fallbacks in place.
- [ ] Fast tier in the loop, frontier tier offline, caching on the prefix.
- [ ] Degradation ladder defined and tested, including voice→chat.
- [ ] Model swaps: shadow → canary (new sessions) → ramp → full; A/B at study level.